# Feature Selection with Random Forest and XGBoost
#### This notebook has 3 sections:
#### 1. Feature selection
#### 2. XGBoost plain
#### 3. XGBoost with cross validation

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# %%
# Load your data here
# Replace the CSV path and target column name with your actual data
data = pd.read_csv("mode_choice_data.csv")
target_col = "travel_mode"

X = data.drop(columns=[target_col])
y = data[target_col] - 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# %% [markdown]
# ## 1. Feature selection

# %%
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_importances_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

rf_importances_df.to_excel("rf_feature_rankings.xlsx", index=False)
rf_importances_df.head(20)

,feature,importance
0,work_accessibility,0.195494
1,shopping_accessibility,0.048597
2,healthcare_access,0.032735
3,cusinter,0.019841
4,id,0.018983
5,age,0.012530
6,crossing_facility,0.011257
7,\tcomfort,0.010959
8,peak_hour_travel,0.010952
9,people,0.010788


In [5]:
y_train.shape

(228,)

# 2. XGBoost plain.

In [9]:
# Feature counts to test
feature_counts = [10, 20, 30, 40, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]

plain_results = []

for n_features in feature_counts:
    # Select top N features from RF importance
    top_features = rf_importances_df["feature"].head(n_features).tolist()
    
    # Model (clean version - no deprecated params)
    model = XGBClassifier(
        objective='multi:softprob',   # multiclass
        num_class=5,                  # 5 classes
        eval_metric="mlogloss",       # correct metric
        random_state=42,
        n_jobs=-1
    )

    # Train
    model.fit(X_train[top_features], y_train)

    # Predict
    y_pred = model.predict(X_test[top_features])

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    
    # Store results
    plain_results.append({
        "num_features": n_features,
        "accuracy": acc
    })

    # Print real-time result
    print(f"[INFO] Features: {n_features} → Accuracy: {acc*100:.2f}%")

    # Save feature importance
    fi_df = pd.DataFrame({
        "feature": top_features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    fi_df.to_excel(f"xgboost_feature_importance_{n_features}.xlsx", index=False)


# Save accuracy results
plain_accuracy_df = pd.DataFrame(plain_results)
plain_accuracy_df.to_excel("xgboost_accuracy_by_num_features.xlsx", index=False)


# Final summary
print("\nFinal Results:")
for row in plain_results:
    print(f"Features: {row['num_features']} → Accuracy: {row['accuracy']*100:.2f}%")


# Best model
best = max(plain_results, key=lambda x: x['accuracy'])
print(f"\n🔥 Best Model → Features: {best['num_features']} | Accuracy: {best['accuracy']*100:.2f}%")

[INFO] Features: 10 → Accuracy: 85.71%
[INFO] Features: 20 → Accuracy: 89.61%
[INFO] Features: 30 → Accuracy: 92.21%
[INFO] Features: 40 → Accuracy: 89.61%
[INFO] Features: 50 → Accuracy: 89.61%
[INFO] Features: 55 → Accuracy: 88.31%
[INFO] Features: 60 → Accuracy: 88.31%
[INFO] Features: 65 → Accuracy: 88.31%
[INFO] Features: 70 → Accuracy: 89.61%
[INFO] Features: 75 → Accuracy: 90.91%
[INFO] Features: 80 → Accuracy: 89.61%
[INFO] Features: 85 → Accuracy: 90.91%
[INFO] Features: 90 → Accuracy: 90.91%
[INFO] Features: 95 → Accuracy: 88.31%
[INFO] Features: 100 → Accuracy: 89.61%

Final Results:
Features: 10 → Accuracy: 85.71%
Features: 20 → Accuracy: 89.61%
Features: 30 → Accuracy: 92.21%
Features: 40 → Accuracy: 89.61%
Features: 50 → Accuracy: 89.61%
Features: 55 → Accuracy: 88.31%
Features: 60 → Accuracy: 88.31%
Features: 65 → Accuracy: 88.31%
Features: 70 → Accuracy: 89.61%
Features: 75 → Accuracy: 90.91%
Features: 80 → Accuracy: 89.61%
Features: 85 → Accuracy: 90.91%
Features: 90 →

# 3. XGBoost and cross validation

In [10]:
cv_results = []

for n_features in feature_counts:
    top_features = rf_importances_df["feature"].head(n_features).tolist()

    model = XGBClassifier(
        objective='multi:softprob',
        num_class=5,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train[top_features],
        y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    mean_acc = scores.mean()

    cv_results.append({
        "num_features": n_features,
        "accuracy": mean_acc
    })

    # 🔥 Print real-time CV result
    print(f"[CV] Features: {n_features} → Accuracy: {mean_acc*100:.2f}%")

    # Fit final model for feature importance
    model.fit(X_train[top_features], y_train)

    fi_df = pd.DataFrame({
        "feature": top_features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    fi_df.to_excel(f"xgboost_cv_feature_importance_{n_features}.xlsx", index=False)


# Save results
cv_accuracy_df = pd.DataFrame(cv_results)
cv_accuracy_df.to_excel("xgboost_cv_accuracy_by_num_features.xlsx", index=False)


# Final summary
print("\nFinal CV Results:")
for row in cv_results:
    print(f"Features: {row['num_features']} → Accuracy: {row['accuracy']*100:.2f}%")


# 🔥 Best model
best = max(cv_results, key=lambda x: x['accuracy'])
print(f"\n🔥 Best CV Model → Features: {best['num_features']} | Accuracy: {best['accuracy']*100:.2f}%")

[CV] Features: 10 → Accuracy: 89.91%
[CV] Features: 20 → Accuracy: 91.25%
[CV] Features: 30 → Accuracy: 91.66%
[CV] Features: 40 → Accuracy: 91.23%
[CV] Features: 50 → Accuracy: 91.23%
[CV] Features: 55 → Accuracy: 91.22%
[CV] Features: 60 → Accuracy: 91.22%
[CV] Features: 65 → Accuracy: 91.66%
[CV] Features: 70 → Accuracy: 91.23%
[CV] Features: 75 → Accuracy: 91.66%
[CV] Features: 80 → Accuracy: 91.66%
[CV] Features: 85 → Accuracy: 91.23%
[CV] Features: 90 → Accuracy: 90.78%
[CV] Features: 95 → Accuracy: 91.22%
[CV] Features: 100 → Accuracy: 91.22%

Final CV Results:
Features: 10 → Accuracy: 89.91%
Features: 20 → Accuracy: 91.25%
Features: 30 → Accuracy: 91.66%
Features: 40 → Accuracy: 91.23%
Features: 50 → Accuracy: 91.23%
Features: 55 → Accuracy: 91.22%
Features: 60 → Accuracy: 91.22%
Features: 65 → Accuracy: 91.66%
Features: 70 → Accuracy: 91.23%
Features: 75 → Accuracy: 91.66%
Features: 80 → Accuracy: 91.66%
Features: 85 → Accuracy: 91.23%
Features: 90 → Accuracy: 90.78%
Features: